In [ ]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import geopandas as gpd
import pandas as pd
import geemap
import os
import pprint

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')


## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'

## threshold for snow in winter to get the 'baseline' value in winter
snow_threshold = 50

## get the time period we care about
start_date_data = '2015-01-01'
end_date_data = '2023-12-31'

# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## dry af
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()
small_square_geom = bounding_box  # Use bounding box directly as the clipping region

## load in the MODIS NDVI data and snow data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD13Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDVI', 'DetailedQA') \
                .map(lambda image: image.clip(small_square_geom))

modis_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.clip(small_square_geom))

## first see the projection of the shapefile
coordidor_projection = md_cooridor.first().geometry().projection()
print('Projection of the cooridor shapefile:', coordidor_projection.getInfo())

## now we can reproject the shapefile
modis_projection = modis_ndvi.first().projection() ## this is the projection of the first image of the modis ndvi data
target_crs = modis_projection.crs() ## this is the crs from the modis.projection object
target_crs_scale = modis_projection.nominalScale() ## this is the scale (confused on what this is, need to figure this out)

## this is a function to reproject it
def reproject_feature(feature):
    return feature.setGeometry(feature.geometry().transform(target_crs, target_crs_scale))

## this is running the function, so now "reprojected_roi" is the new object we care about
reprojected_roi = md_cooridor.map(reproject_feature)

## make sure the time property is there
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )

## make sure the times are good
modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))

print("Number of images in the ndvi collection:", modis_ndvi.size().getInfo())
print("Number of images in the snow collection:", modis_snow.size().getInfo())


In [ ]:
def generate_snow_mask(ndvi_image, modis_snow, snow_threshold):
    """
    Generates a snow mask for the given NDVI image by finding the closest snow image.
    """
    ## get the date of the NDVI image
    image_date = ee.Date(ndvi_image.get('system:time_start'))

    ## define a buffer period to search for snow images
    buffer_period = 16  # e.g., 16 days buffer

    ## pull out the images that are within the buffer
    snow_images = modis_snow.filterDate(image_date.advance(-buffer_period, 'day'), image_date.advance(buffer_period, 'day')).mean()

    ## if the snow cover layer is greater than or equal to the threshold, create a snow mask
    snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)
    return snow_mask


## visualie snow mask

def visualize_snow_mask(ndvi_image, modis_snow, snow_threshold):
    """
    Generates and returns a snow mask for the given NDVI image and MODIS snow data.
    """
    # Ensure that the ndvi_image is not None
    if not ndvi_image:
        raise ValueError("The NDVI image is not available.")
    
    snow_mask = generate_snow_mask(ndvi_image, modis_snow, snow_threshold)
    return snow_mask

## use a slightly larger date range to ensure you get an image
target_date = '2015-08-01'
target_end_date = '2015-08-10'  # Allow a small range to ensure data availability
target_date_gee = ee.Date(target_date)

## filter the NDVI collection for the given date range
filtered_image_collection = modis_snow.filterDate(target_date, target_end_date)

## check if the filtered image collection contains images
if filtered_image_collection.size().getInfo() == 0:
    raise ValueError(f"No images found for the given date range: {target_date} to {target_end_date}")

## get the first image in that range
specific_image = filtered_image_collection.first()

## check if the specific image is valid
if not specific_image:
    raise ValueError("No valid NDVI image found for the specified date.")

## export or visualize snow mask for a specific date
snow_mask_example = visualize_snow_mask(specific_image, modis_snow, snow_threshold)

## ensure that the snow mask is not None
if not snow_mask_example:
    raise ValueError("The snow mask could not be generated.")

## export the snow mask as needed
def export_snow_mask_example(snow_mask_example, file_name, folder_name, region, scale=500):
    """
    Exports the snow mask as a GeoTIFF to Google Drive.
    
    Parameters:
    - snow_mask_example: The snow mask image to be exported.
    - file_name: The name of the file to save in Google Drive.
    - folder_name: The folder in Google Drive to save the file.
    - region: The geometry/region to export (should be the area of interest).
    - scale: The resolution in meters (default: 500).
    """

    try:
        task = ee.batch.Export.image.toDrive(
            image=snow_mask_example.toFloat(),  # Export as a floating-point image
            description=f'{file_name}_export',  # Task description
            folder=folder_name,  # Folder name in Google Drive
            fileNamePrefix=file_name,  # Prefix for the file name
            region=region,  # The region of interest (bounding box or geometry)
            scale=scale,  # Spatial resolution in meters
            maxPixels=1e13,  # Ensure the export doesn't fail due to pixel limits
            fileFormat='GeoTIFF'  # File format for export
        )

        # Start the export task
        task.start()
        print(f'Exporting {file_name} to Google Drive...')

    except Exception as e:
        print(f'Error exporting {file_name}: {e}')


## call the function to export the snow mask example
export_snow_mask_example(
    snow_mask_example=snow_mask_example,  # This is the snow mask you want to export
    file_name='snow_mask_example',  # Change to your desired file name
    folder_name='DroughtSurfing',  # Folder in Google Drive where it will be saved
    region=small_square_geom,  # Region of interest (geometry for export)
    scale=500  # Resolution in meters (adjust if necessary)
)
